# Tạo VOI trung gian và tổng hợp 3 kênh
Chạy trên CPU. Bước đầu tái tạo VOI theo TV1 từ ảnh gốc; mỗi mask được đối chiếu với bộ TV1 trước khi lưu. Sau đó tạo 3 kênh và QC. Có thể chạy lại để tiếp tục nếu Colab ngắt.

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/CS406-SCB-TumorSeg')
TV2 = PROJECT / 'TV2_Tuan1'
DATA = PROJECT / 'data_real'
assert (TV2 / 'export_voi.py').is_file(), f'Không tìm thấy mã TV2: {TV2}'
assert (DATA / 'splits/splits_metadata.csv').is_file(), f'Không tìm thấy dữ liệu: {DATA}'


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(TV2 / 'requirements.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests'], cwd=TV2, check=True)


In [ ]:
# Tạo VOI; xem data_real/tv2_step3/voi_export.csv nếu có lỗi.
subprocess.run([sys.executable, str(TV2 / 'export_voi.py'), '--data-root', str(DATA)], cwd=TV2, check=True)


In [ ]:
# Chỉ chạy sau khi tất cả VOI được tạo thành công.
subprocess.run([sys.executable, str(TV2 / 'run_step3.py'), '--data-root', str(DATA), '--mode', 'recompute'], cwd=TV2, check=True)


In [ ]:
import json
reports = DATA / 'tv2_step3/reports'
print(json.dumps(json.loads((reports / 'provenance.json').read_text()), indent=2, ensure_ascii=False))
print(json.dumps(json.loads((reports / 'qc_summary.json').read_text()), indent=2, ensure_ascii=False))
print('Báo cáo:', reports)
